# Konnon Drawing Model - 116.4M Scaled Training on Real Quick, Draw! Data
This notebook runs the 116.4M Parameter Staged Visual Stroke Transformer on Kaggle GPU.

In [ ]:
# ======================================================================
# CELL 1: SETUP WORKING ENVIRONMENT & PULL CODEBASE
# ======================================================================
import os, sys, shutil

%cd /kaggle/working
if os.path.exists("/kaggle/working/tamu"):
    %cd /kaggle/working/tamu
    !git fetch origin main
    !git reset --hard origin/main
else:
    !git clone https://github.com/Kannangog/tamu.git
    %cd /kaggle/working/tamu

%env PYTHONPATH=/kaggle/working/tamu/src
!pip install -e . --no-deps
!nvidia-smi

In [ ]:
# ======================================================================
# CELL 2: INGEST REAL QUICK, DRAW! STAGED DATASET (20+ CATEGORIES)
# ======================================================================
!python scripts/ingest_quickdraw_staged.py \
    --drawings_per_cat 600 \
    --max_seq_len 512 \
    --canvas_res 128 \
    --out_dir data/processed/quickdraw_staged_50k

In [ ]:
# ======================================================================
# CELL 3: TRAIN SCALED 116.4M PARAMETER TRANSFORMER (GPU FP16)
# ======================================================================
!python scripts/train_scaled_116m.py \
    --device cuda \
    --fp16 \
    --batch_size 32 \
    --grad_accum 4 \
    --epochs 10 \
    --lr 3e-4 \
    --d_model 768 \
    --n_layers 12 \
    --n_heads 12 \
    --d_ff 3072 \
    --max_seq_len 512 \
    --canvas_res 128 \
    --data_dir data/processed/quickdraw_staged_50k \
    --out_dir outputs_v2/scaled_116m

In [ ]:
# ======================================================================
# CELL 4: PACKAGE CHECKPOINT & PREVIEW VISUAL DRAWINGS
# ======================================================================
import os, zipfile
from IPython.display import Image, display

# Preview the generated visual test drawing
if os.path.exists("outputs_v2/scaled_116m/test_latest.png"):
    print("🎨 Model's Generated Test Drawing from Epoch 10:")
    display(Image(filename="outputs_v2/scaled_116m/test_latest.png"))

# Compress checkpoint for fast download
print("📦 Compressing checkpoint...")
ckpt_path = "outputs_v2/scaled_116m/checkpoint_best.pt"
zip_path = "/kaggle/working/checkpoint_116m_best.zip"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(ckpt_path, arcname="checkpoint_best.pt")

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"✅ Checkpoint compressed: {zip_path} ({size_mb:.2f} MB)")
print("👉 Download checkpoint_116m_best.zip directly from the Kaggle Output panel!")